# Week 11 Lab: Words as Features

Use the demo as a reference. Most of the code is supplied. Replace each `___` with the missing code, run the cell, and check the output. Then answer the explanation prompts using your own results. Estimated time: 60 to 90 minutes. Use DATA_GUIDE.txt and your existing .venv.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

## Part 1: Inspect the text

Load the data and check the text for missing, blank, and unusable values. Complete the blanks: count missing text with `.isna()`, and count messages whose token count is `0`.

In [ ]:
raw_df = pd.read_csv('data/sms_messages.csv')
print('Raw shape:', raw_df.shape)
display(raw_df['is_spam'].value_counts().sort_index().rename(index={0: 'ham', 1: 'spam'}))

analyzer = CountVectorizer().build_analyzer()

def token_count(text):
    return len(analyzer(text))

text = raw_df['text']
missing_text = text.___()                      # which rows have missing text?
blank_text = text.fillna('').str.strip().eq('')
no_tokens = ~missing_text & ~blank_text & text.fillna('').map(token_count).eq(___)
print('Missing text:', int(missing_text.sum()))
print('Blank or whitespace-only text:', int(blank_text.sum()))
print('Text with no usable tokens:', int(no_tokens.sum()))
display(raw_df.loc[no_tokens, ['record_id', 'text', 'is_spam']])

**Questions:** (a) Why could accuracy alone mislead for this dataset? Use the class counts. (b) What will the vectorizer produce for the no-token messages shown above, and why does DATA_GUIDE.txt keep them?

**Your explanation:**

Replace this text with your response.

## Part 2: Prepare, add numeric features, and split

The validity and duplicate checks are supplied. Complete `add_message_features` using the definitions in DATA_GUIDE.txt (hint: `.str.len()` and `.str.count(r'\d')`), then complete the stratified split.

In [ ]:
working_df = raw_df.copy()
invalid = (working_df['text'].isna() | working_df['text'].str.strip().eq('')
           | ~working_df['is_spam'].isin([0, 1]))
rejected_ids = working_df.loc[invalid, 'record_id'].tolist()
working_df = working_df.loc[~invalid].copy()
working_df['duplicate_key'] = working_df['text'].str.strip().str.lower()
assert working_df.groupby('duplicate_key')['is_spam'].nunique().le(1).all(), 'Investigate conflicting labels.'
duplicates = working_df.duplicated('duplicate_key', keep='first')
duplicate_ids = working_df.loc[duplicates, 'record_id'].tolist()
working_df = working_df.loc[~duplicates].copy()
print('Rejected:', len(rejected_ids), 'Duplicates removed:', len(duplicate_ids), 'Retained:', len(working_df))

def add_message_features(df):
    out = df.copy()
    text = out['text'].fillna('').str.strip()
    out['char_count'] = ___        # number of characters
    out['digit_count'] = ___       # number of digits
    return out

working_df = add_message_features(working_df)

TEXT_COLUMN = 'text'
NUMERIC_COLUMNS = ['char_count', 'digit_count']
FEATURES = [TEXT_COLUMN] + NUMERIC_COLUMNS
train_df, test_df = train_test_split(working_df, test_size=0.20, random_state=42,
                                     stratify=___)
X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train, y_test = train_df['is_spam'], test_df['is_spam']
assert set(train_df['duplicate_key']).isdisjoint(test_df['duplicate_key'])
print('Training rows:', len(X_train), 'Test rows:', len(X_test))
display(train_df.groupby('is_spam')[NUMERIC_COLUMNS].median())

**Question:** char_count and digit_count are calculated before the split, but the vocabulary is learned after it. Explain why the first is not leakage and the second would be.

**Your explanation:**

Replace this text with your response.

## Part 3: Explain the representation

Create a count matrix for three invented documents, then compare default tokens with English stop-word removal. Keep this teaching vectorizer separate from the SMS model.

In [ ]:
docs = ['not a good phone', 'good phone good', 'not reliable']
teach = CountVectorizer()
matrix = teach.___(docs)                 # learn the vocabulary and build the matrix
display(pd.DataFrame(matrix.toarray(), columns=teach.get_feature_names_out(), index=docs))
print('Default tokens:', CountVectorizer().build_analyzer()('not a good phone'))
print('With stop words removed:', CountVectorizer(stop_words=___).build_analyzer()('not a good phone'))

**Question:** What happened to "a"? What does the value 2 in the matrix mean? Why does losing "not" matter?

**Your explanation:**

Replace this text with your response.

## Part 4: Add a text branch to a provided transformer

The numeric branch is supplied. Add the text branch: name it `'text'`, use the `text_prep` pipeline, and give it the text column. Then complete the TF-IDF settings used in the demo: `lowercase=True, stop_words=None, ngram_range=(1, 1), min_df=2`. Fit using training rows only. The assertion checks that you passed the column correctly.

In [ ]:
def fill_missing_text(text):
    """Replace missing documents with empty strings so the vectorizer can read every row."""
    return pd.Series(text).fillna('')


text_prep = Pipeline([
    ('fill_missing', FunctionTransformer(fill_missing_text, feature_names_out='one-to-one')),
    ('tfidf', TfidfVectorizer(lowercase=___, stop_words=___, ngram_range=___, min_df=___)),
])
preprocessor = ColumnTransformer([
    (___, ___, ___),                                   # the text branch
    ('numeric', StandardScaler(), NUMERIC_COLUMNS),
])
model = Pipeline([('prep', preprocessor),
                  ('classifier', LogisticRegression(max_iter=1000, random_state=42))])
model.fit(___, ___)

train_features = model.named_steps['prep'].transform(X_train)
assert train_features.shape[0] == len(X_train), 'Check how the text column is named in the text branch.'
print('Fitted on training rows only.')

**Question:** Why is the text column given as `'text'` rather than `['text']`? Why does the text branch fill missing values before TfidfVectorizer, even though Part 1 found no missing text?

**Your explanation:**

Replace this text with your response.

## Part 5: Inspect the learned features

Complete the blanks to report the transformed shapes, the number of text features, and stored nonzero values, and to show selected terms from the fitted vocabulary.

In [ ]:
fitted_prep = model.named_steps['prep']
fitted_tfidf = fitted_prep.named_transformers_[___].named_steps['tfidf']
test_features = fitted_prep.___(X_test)          # apply, do not refit
feature_names = fitted_prep.get_feature_names_out()
print('Training shape:', train_features.shape, ' Test shape:', test_features.shape)
print('Text features:', len(fitted_tfidf.get_feature_names_out()))
print('Stored nonzero training values:', train_features.___)
print('Selected terms:', [t for t in ['free', 'prize', 'call', 'meet', 'tomorrow', 'not'] if t in fitted_tfidf.vocabulary_])
print('Last 2 feature names:', feature_names[-2:].tolist())

**Question:** What does one row and one column of the transformed matrix represent? Why do the training and test matrices have the same number of columns?

**Your explanation:**

Replace this text with your response.

## Part 6: Evaluate with a baseline

Complete the baseline strategy and the confusion-matrix call. Spam (1) is the positive class.

In [ ]:
baseline = DummyClassifier(strategy=___).fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
pred = model.predict(X_test)

def scores(y_true, y_pred):
    return [accuracy_score(y_true, y_pred), precision_score(y_true, y_pred, zero_division=0),
            recall_score(y_true, y_pred, zero_division=0)]

display(pd.DataFrame({'model': scores(y_test, pred), 'majority_baseline': scores(y_test, baseline_pred)},
                     index=['accuracy', 'precision (spam)', 'recall (spam)']).round(3))
cm = confusion_matrix(___, ___, labels=[0, 1])
display(pd.DataFrame(cm, index=['actual ham', 'actual spam'], columns=['predicted ham', 'predicted spam']))

**Question:** Does the model beat the baseline? Use your numbers. Explain a false positive and a false negative in this setting and which would worry you more.

**Your explanation:**

Replace this text with your response.

## Part 7: Test on new text

These invented rows include missing text, blank text, and an unknown-only word. Prepare them with the same feature function, then predict with the fitted pipeline. Complete the blanks.

In [ ]:
new_df = pd.DataFrame({'record_id': ['NEW1', 'NEW2', 'NEW3', 'NEW4'],
                       'text': ['Are you free for lunch today?', None, '   ', 'zzzxxyyqq']})
vocab_size_before = len(fitted_tfidf.get_feature_names_out())
new_ready = ___(new_df)                          # same feature function as training
new_result = new_ready[['record_id', 'text', 'char_count', 'digit_count']].copy()
new_result['recognized_terms'] = fitted_tfidf.transform(fill_missing_text(new_ready['text'])).getnnz(axis=1)
new_result['predicted_is_spam'] = model.___(new_ready[FEATURES])
display(new_result)
assert len(fitted_tfidf.get_feature_names_out()) == vocab_size_before
print('Vocabulary unchanged:', vocab_size_before)

**Question:** What would have happened to NEW2 without the fill step? Rows NEW2 to NEW4 have no recognized terms but still get a prediction. Where does that prediction come from, and should we trust it?

**Your explanation:**

Replace this text with your response.

## Part 8: Reflection

Answer briefly in your own words.

1. What made sense this week?
2. What was difficult or unclear?
3. What do you still have questions about?
4. What would you check first if this text workflow failed on new data?

Restart the kernel, run all cells, and save with outputs before you commit and push.

**Your reflection:**

Replace this text with your response.